# 02 · 샘플 기술/노하우 데이터 주입

**파이프라인 2/4** — 파이프라인을 테스트할 수 있도록 실제 Teams/Mail에 샘플 기술·노하우
메시지/메일을 **write 도구로 전송**합니다.

**모델/에이전트 없이** MCP write 도구를 **직접 호출(`call_tool`)** 합니다. 데이터를 밀어 넣는
단순 작업이므로 LLM 루프가 필요 없습니다. 아래에서 도구 이름을 자동 감지(수정 가능)한 뒤
샘플을 순회하며 그대로 전송합니다.

> ⚠️ 실제 메일/메시지가 전송됩니다. 수신자/대상은 아래 `MAIL_TO` / `TEAMS_TO`에서 지정하세요.

## 셋업 (설정 + 인증 + MCP 헬퍼)

In [ ]:
# ============================================================
# 셋업 — 이 셀 하나로 설정·인증 준비 (pipeline/*.py 없이 독립 실행)
# ============================================================
import os, json, asyncio
from pathlib import Path
from contextlib import AsyncExitStack

import msal
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# 1) 프로젝트 루트를 찾아 .env 로드 (.env.example/requirements.txt가 있는 폴더)
def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / '.env.example').is_file() or (base / 'requirements.txt').is_file():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')
TOKEN_CACHE = PROJECT_ROOT / '.token_cache.json'   # 로그인 1회 → 이후 노트북/웹앱이 재사용

# 2) 설정값 (.env에서 읽음)
TENANT_ID     = os.environ.get('TENANT_ID') or '00000000-0000-0000-0000-000000000000'
CLIENT_ID     = os.environ.get('CLIENT_ID', '')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')
AUTH_MODE     = os.environ.get('AUTH_MODE', 'device_code')   # device_code | client_credentials
AUTHORITY     = f'https://login.microsoftonline.com/{TENANT_ID}'

def _mcp_url(env_var, server_id):
    return (os.environ.get(env_var) or '').strip() or \
        f'https://agent365.svc.cloud.microsoft/agents/tenants/{TENANT_ID}/servers/{server_id}'

# 두 MCP 소스: Mail / Teams. 서버마다 delegated 스코프가 달라 토큰도 소스별로 1개씩.
SOURCES = {
    'mail':  {'label': 'Mail',  'url': _mcp_url('MAIL_MCP_SERVER_URL',  'mcp_MailTools')},
    'teams': {'label': 'Teams', 'url': _mcp_url('TEAMS_MCP_SERVER_URL', 'mcp_TeamsServer')},
}
for _s in SOURCES.values():
    _s['scopes'] = [f"{_s['url']}/.default"]

# 3) MSAL 인증 — 소스별 access token. device_code는 최초 1회만 로그인, 이후 캐시 재사용.
_cache = msal.SerializableTokenCache()
if TOKEN_CACHE.exists():
    _cache.deserialize(TOKEN_CACHE.read_text(encoding='utf-8'))

def _save_cache():
    if _cache.has_state_changed:
        TOKEN_CACHE.write_text(_cache.serialize(), encoding='utf-8')

def ensure_token(source_key: str) -> str:
    """소스 1개의 access token 확보 (없으면 device-code 로그인)."""
    assert CLIENT_ID, 'CLIENT_ID가 .env에 없습니다 (.env.example 참고).'
    scopes = SOURCES[source_key]['scopes']
    if AUTH_MODE == 'client_credentials':           # 앱 전용 토큰
        app = msal.ConfidentialClientApplication(
            CLIENT_ID, authority=AUTHORITY, client_credential=CLIENT_SECRET, token_cache=_cache)
        res = app.acquire_token_for_client(scopes=scopes)
    else:                                           # 위임(사용자) 로그인
        app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=_cache)
        accounts = app.get_accounts()
        res = app.acquire_token_silent(scopes, account=accounts[0]) if accounts else None
        if not (res and 'access_token' in res):     # 캐시에 없으면 device-code 로그인
            flow = app.initiate_device_flow(scopes=scopes)
            print(flow['message'])                   # https://microsoft.com/devicelogin + 코드 안내
            res = app.acquire_token_by_device_flow(flow)
    _save_cache()
    if not res or 'access_token' not in res:
        raise RuntimeError((res or {}).get('error_description', 'access token 획득 실패'))
    return res['access_token']

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TENANT_ID   :', TENANT_ID, '| CLIENT_ID set:', bool(CLIENT_ID), '| AUTH_MODE:', AUTH_MODE)
for _k, _s in SOURCES.items():
    print(f"  [{_k}] {_s['label']} -> {_s['url']}")

In [ ]:
# --- MCP 읽기 헬퍼: 연결은 호출할 때마다 열고 항상 닫는다 ---
async def _with_session(source_key, token, fn):
    url = SOURCES[source_key]['url']
    async with streamablehttp_client(url, headers={'Authorization': f'Bearer {token}'}) as (r, w, _):
        async with ClientSession(r, w) as s:      # MCP 세션 초기화(handshake)
            await s.initialize()
            return await fn(s)

async def list_tools(token, source_key):
    """MCP 서버가 노출하는 도구(함수) 목록."""
    res = await _with_session(source_key, token, lambda s: s.list_tools())
    return list(res.tools or [])

async def call_tool(token, source_key, name, args=None):
    """MCP 도구를 이름+인자로 직접 호출."""
    return await _with_session(source_key, token, lambda s: s.call_tool(name, arguments=args or {}))

def tool_schema(tool):
    sc = getattr(tool, 'inputSchema', None) or {'type': 'object', 'properties': {}}
    return sc.model_dump() if hasattr(sc, 'model_dump') else sc

def content_to_text(result) -> str:
    """MCP 도구 결과의 content 블록들을 읽기 좋은 문자열로 평탄화."""
    parts = []
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text':
            parts.append(getattr(b, 'text', ''))
        else:
            data = b.model_dump() if hasattr(b, 'model_dump') else b
            parts.append('```json\n' + json.dumps(data, ensure_ascii=False, indent=2, default=str) + '\n```')
    text = '\n'.join(parts)
    return f'ERROR: {text}' if getattr(result, 'isError', False) else text

## 1. 토큰 확보 + 툴 목록 (01에서 캐시된 로그인 재사용)

In [ ]:
tokens = {}
for key in SOURCES:
    try:
        tokens[key] = ensure_token(key)
    except Exception as exc:
        print(f'{key} 토큰 실패: {exc}')
print('확보된 소스:', list(tokens.keys()))

# 각 소스의 MCP 도구 목록 — 전송에 쓸 write 도구 이름을 여기서 확인
tools_by_source = {}
for key, token in tokens.items():
    tools_by_source[key] = await list_tools(token, key)
    print(f'{key}: ' + ', '.join(t.name for t in tools_by_source[key]))

## 2. 보낼 샘플 콘텐츠 + 대상 정의
실제 위키에 담길 법한 재사용 가능한 기술 지식/노하우 예시입니다. 자유롭게 편집하세요.

In [ ]:
# 전송 대상 (직접 호출용).
# MAIL_TO 를 비우면 로그인한 '본인(발신자)'에게 셀프 발송합니다. 특정 대상에게 보내려면 주소를 넣으세요.
MAIL_TO  = ''     # 비우면 셀프 발송 / 예: 'someone@yourtenant.onmicrosoft.com'
TEAMS_TO = ''     # 비우면 'Notes to Self'(본인) 채팅으로 발송 / 특정 채팅은 chatId 지정

SAMPLE_EMAILS = [
    {'subject': '[노하우] Postgres 커넥션 풀 튜닝 정리',
     'body': ('PgBouncer를 transaction 모드로 두고 app 쪽 pool_size는 CPU 코어수*2로 제한. '
              'idle_in_transaction_session_timeout=30s로 좀비 커넥션 방지. '
              'max_connections를 무작정 늘리기보다 PgBouncer로 멀티플렉싱하는 게 안정적이었음.')},
    {'subject': '[트러블슈팅] k8s 파드 CrashLoopBackOff 원인 추적법',
     'body': ('kubectl describe pod로 이벤트 확인 -> kubectl logs --previous로 직전 컨테이너 로그. '
              'OOMKilled면 requests/limits 재조정. readinessProbe 타임아웃이 잦으면 initialDelaySeconds 상향.')},
]

SAMPLE_TEAMS_MESSAGES = [
    ('[팁] GitHub Actions 캐시 히트율 올리기: actions/cache로 node_modules + ~/.npm 둘 다 '
     '키에 lockfile 해시를 포함시키면 히트율이 크게 올라감.'),
    ('[결정사항] 사내 로그 표준: 구조적 로깅(JSON) + trace_id 필수. 레벨은 ERROR/WARN/INFO만, '
     'DEBUG는 로컬 전용.'),
]
print('메일', len(SAMPLE_EMAILS), '건, Teams', len(SAMPLE_TEAMS_MESSAGES), '건 준비됨')

## 3. 전송할 MCP 도구 선택 (자동 감지 · 수정 가능)
위 툴 목록에서 write 도구를 고릅니다. 자동 감지가 틀리면 이름을 직접 지정하세요.
출력된 **입력 스키마**를 보고 4·5단계의 인자 이름이 맞는지 확인하면 됩니다.

In [ ]:
# 전송에 쓸 도구 이름. 비우면 이름 패턴으로 자동 감지.
MAIL_SEND_TOOL  = ''
TEAMS_SELF_TOOL = ''   # 본인에게(Notes to Self) — chatId 불필요
TEAMS_CHAT_TOOL = ''   # 특정 chatId로 보낼 때

def _auto(source, include, exclude=()):
    names = [t.name for t in tools_by_source.get(source, [])]
    def score(n):
        nl = n.lower()
        if any(x in nl for x in exclude):
            return -1
        return sum(1 for k in include if k in nl)
    best = max(names, key=score, default='')
    return best if best and score(best) > 0 else ''

MAIL_SEND_TOOL  = (MAIL_SEND_TOOL  or _auto('mail', ['send', 'email'], ['draft', 'reply', 'forward'])
                   or _auto('mail', ['send'], ['draft', 'reply']))
# 셀프 메시지: 'self' 포함 도구(SendMessageToSelf) 우선. chatId 불필요.
TEAMS_SELF_TOOL = (TEAMS_SELF_TOOL or _auto('teams', ['message', 'self'], ['file'])
                   or _auto('teams', ['send', 'self'], ['file']))
# 특정 채팅: message+chat (파일/채널/self 제외)
TEAMS_CHAT_TOOL = (TEAMS_CHAT_TOOL or _auto('teams', ['send', 'message', 'chat'],
                                            ['file', 'reaction', 'reply', 'channel', 'self', 'user']))
print('메일 전송 도구   :', MAIL_SEND_TOOL or '(자동 감지 실패 — 위 목록에서 수동 지정)')
print('Teams 셀프 도구  :', TEAMS_SELF_TOOL or '(없음)')
print('Teams 채팅 도구  :', TEAMS_CHAT_TOOL or '(없음)')

# 논리 인자(subject/body/message/to/target)를 실제 스키마의 파라미터 이름으로 매핑
_SYN = {
    'subject': ['subject', 'title', 'summary'],
    'body':    ['body', 'content', 'message', 'text', 'bodyText', 'messageBody', 'contentBody'],
    'message': ['message', 'content', 'text', 'body', 'messageBody'],
    'to':      ['to', 'toRecipients', 'recipients', 'recipient', 'toRecipient', 'emailAddress', 'address'],
    'target':  ['chatId', 'channelId', 'conversationId', 'threadId', 'targetId', 'recipient', 'to'],
}
def _fit(source, tool_name, logical):
    """논리 인자를 실제 스키마 파라미터 이름 + 타입에 맞춰 구성."""
    t = next((x for x in tools_by_source.get(source, []) if x.name == tool_name), None)
    props = tool_schema(t).get('properties', {}) if t else {}

    def _shape(addr, spec):
        # 이메일 주소/대상을 스키마 타입에 맞게 감쌈 (Graph식 toRecipients 등)
        typ = (spec or {}).get('type')
        if typ == 'array':
            items = spec.get('items', {}) if isinstance(spec.get('items'), dict) else {}
            iprops = items.get('properties', {}) if isinstance(items, dict) else {}
            if 'emailAddress' in iprops:
                return [{'emailAddress': {'address': addr}}]
            if 'address' in iprops:
                return [{'address': addr}]
            return [addr]
        if typ == 'object':
            oprops = spec.get('properties', {})
            if 'emailAddress' in oprops:
                return {'emailAddress': {'address': addr}}
            if 'address' in oprops:
                return {'address': addr}
        return addr

    out = {}
    for lk, val in logical.items():
        cand = next((c for c in _SYN.get(lk, [lk]) if (not props or c in props)), None) or lk
        spec = props.get(cand, {}) if props else {}
        if lk in ('to', 'target') and isinstance(val, str):
            val = _shape(val, spec)
        out[cand] = val
    return out

import base64
def signed_in_email(token: str = '') -> str:
    """로그인한 사용자(발신자)의 이메일. device_code면 MSAL 계정에서, 아니면 토큰 클레임에서 추출."""
    try:
        app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=_cache)
        accts = app.get_accounts()
        if accts and accts[0].get('username'):
            return accts[0]['username']
    except Exception:
        pass
    if token:                       # access token(JWT) 클레임에서 추출 (검증 없이 디코드만)
        try:
            payload = token.split('.')[1]
            payload += '=' * (-len(payload) % 4)
            claims = json.loads(base64.urlsafe_b64decode(payload))
            for k in ('preferred_username', 'upn', 'unique_name', 'email'):
                if claims.get(k):
                    return claims[k]
        except Exception:
            pass
    return ''


# 선택된 도구의 입력 스키마 출력 (인자 이름 참고)
for src, name in (('mail', MAIL_SEND_TOOL), ('teams', TEAMS_SELF_TOOL), ('teams', TEAMS_CHAT_TOOL)):
    t = next((x for x in tools_by_source.get(src, []) if x.name == name), None)
    if t:
        print(f'\n[{src}] {name} 입력 스키마:')
        print(json.dumps(tool_schema(t), ensure_ascii=False, indent=2)[:1500])

## 4. 메일 샘플 전송 (Mail MCP 직접 호출)
`call_tool`로 각 메일을 그대로 전송합니다. 필드명이 스키마와 다르면 위 `_SYN`을 조정하세요.

In [ ]:
# MAIL_TO가 비어 있으면 로그인한 본인(발신자)에게 셀프 발송
recipient = MAIL_TO or signed_in_email(tokens.get('mail', ''))
if not recipient:
    print('⚠ 수신자를 확인할 수 없습니다 — MAIL_TO에 본인 이메일을 직접 넣어주세요.')
elif not (tokens.get('mail') and MAIL_SEND_TOOL):
    print('mail 토큰/전송 도구가 없어 건너뜀 (3단계 스키마로 도구·인자명을 맞추세요)')
else:
    if not MAIL_TO:
        print(f'MAIL_TO가 비어 있어 본인({recipient})에게 셀프 발송합니다.')
    for e in SAMPLE_EMAILS:
        args = _fit('mail', MAIL_SEND_TOOL, {'subject': e['subject'], 'body': e['body'], 'to': recipient})
        print(f"\n[전송] {MAIL_SEND_TOOL} → {recipient} (args keys: {list(args)})")
        try:
            result = await call_tool(tokens['mail'], 'mail', MAIL_SEND_TOOL, args)
            print('  결과:', content_to_text(result)[:1500] or '(빈 응답)')
        except Exception as exc:
            print('  실패:', exc)
    print('\n※ 결과에 draft/messageId만 있고 받은편지함에 없으면, 자동 감지된 도구가 실제 발송이 아닌')
    print('  임시보관함 저장 도구일 수 있습니다. 3단계 목록에서 send 계열 도구로 MAIL_SEND_TOOL을 지정하세요.')

## 5. Teams 샘플 게시 (Teams MCP 직접 호출)
`TEAMS_TO`가 비어 있으면 **본인 'Notes to Self' 채팅**(`SendMessageToSelf`, chatId 불필요)으로 보냅니다.
특정 채팅에 보내려면 2단계 `TEAMS_TO`에 chatId를 넣으세요(`SendMessageToChat`).

In [ ]:
# TEAMS_TO 비어 있으면 셀프(Notes to Self), 있으면 해당 chatId로
teams_tool = TEAMS_CHAT_TOOL if TEAMS_TO else TEAMS_SELF_TOOL
if not (tokens.get('teams') and teams_tool):
    print('teams 토큰/게시 도구가 없어 건너뜀 (3단계 목록에서 도구명을 확인하세요)')
else:
    dest = f'채팅 {TEAMS_TO}' if TEAMS_TO else 'Notes to Self(본인)'
    print(f'게시 대상: {dest} · 도구: {teams_tool}')
    for m in SAMPLE_TEAMS_MESSAGES:
        args = _fit('teams', teams_tool, {'message': m})   # message -> content 매핑
        if TEAMS_TO:
            args['chatId'] = TEAMS_TO
        print(f"\n[게시] {teams_tool} (args keys: {list(args)})")
        try:
            result = await call_tool(tokens['teams'], 'teams', teams_tool, args)
            print('  결과:', content_to_text(result)[:1500] or '(빈 응답)')
        except Exception as exc:
            print('  실패:', exc)

## 6. 주입 결과 확인 (Outlook / Teams 접속 URL)
전송된 샘플을 브라우저에서 바로 확인할 수 있는 링크입니다. (인덱싱/동기화 지연이 있을 수 있음)

**Outlook (메일)**
- 받은 편지함: <https://outlook.office.com/mail/inbox>
- 보낸 편지함: <https://outlook.office.com/mail/sentitems>

**Teams (메시지)**
- Teams 웹: <https://teams.microsoft.com/>
- 채팅 목록: <https://teams.microsoft.com/v2/#/conversations>

> 로그인 계정은 노트북 `01`에서 device-code로 로그인한 M365 계정과 동일해야 합니다.

In [ ]:
# 확인용 URL 출력 (클릭해서 이동)
LINKS = {
    'Outlook 받은 편지함': 'https://outlook.office.com/mail/inbox',
    'Outlook 보낸 편지함': 'https://outlook.office.com/mail/sentitems',
    'Teams 웹':           'https://teams.microsoft.com/',
    'Teams 채팅 목록':     'https://teams.microsoft.com/v2/#/conversations',
}
print('전송한 샘플을 아래에서 확인하세요:')
for label, url in LINKS.items():
    print(f'  - {label}: {url}')

try:                       # Jupyter면 클릭 가능한 링크로도 표시
    from IPython.display import display, Markdown
    display(Markdown('\n'.join(f'- **{k}**: [{v}]({v})' for k, v in LINKS.items())))
except Exception:
    pass

✅ 샘플이 전송되었습니다. 위 URL에서 확인한 뒤, **03_fetch_data**에서 MCP로 조회해 보세요.